# Quick exploration into `everef` data

Scratch pad for configuring `dlt` pipeline with `everef` data

### Imports

In [1]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv
from dlt.extract import DltResource

from datetime import date, datetime, timedelta, UTC

from collections.abc import Generator, Iterator

# import requests
from dlt.sources.helpers import requests
from requests import codes

import logging.config

from email.utils import parsedate_to_datetime

import pandas as pd
import pyarrow as pa

### Sources

In [2]:
BASE_URL = "https://data.everef.net/market-history"
destination = "file://../.local/notebook/data/bronze"

totals_file = f"{BASE_URL}/totals.json"

In [3]:
import pathlib
import json
import atexit

# https://www.youtube.com/watch?v=9L77QExPmI0
logger = logging.getLogger("eve_market_dlt")
logging_config = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "simple": {
            "format": "%(levelname)s: %(message)s"
        },
        "detailed": {
            "format": "[%(levelname)s] %(asctime)s | %(module)s:L%(lineno)d : %(message)s",
            "datefmt": "%Y-%m-%dT%H:%M:%S%z"
        }
    },
    "handlers": {
        "stderr": {
            "class": "logging.StreamHandler",
            "level": "WARNING",
            "formatter": "detailed",
            "stream": "ext://sys.stderr"
        },
        # "file": {
        #     "class": "logging.handlers.RotatingFileHandler",
        #     "level": "DEBUG",
        #     "formatter": "simple",
        #     "filename": "logs/tmp.log",
        #     "maxBytes": 10000,
        #     "backupCount": 3
        # },
        "queue_handler": {
            "class": "logging.handlers.QueueHandler",
            "respect_handler_level": True,
            "handlers": [
                "stderr",
                # "file"
            ]
        }
    },
    "loggers": {
        "root": {
            "level": "DEBUG",
            "handlers": [
                "queue_handler"
            ]
        }
    }
}

def setup_logging():
    # config_file = pathlib.Path("logging_configs/config.json")
    # with open(config_file) as f_in:
    #     config = json.load(f_in)
    # logging.config.dictConfig(config)
    logging.config.dictConfig(logging_config)
    queue_handler = logging.getHandlerByName("queue_handler")
    if queue_handler is not None:
        queue_handler.listener.start()
        atexit.register(queue_handler.listener.stop)

setup_logging()

In [4]:
def get_date(filename: str) -> date:
    date_str = filename.removeprefix("market-history-") \
                        .removesuffix(".csv.bz2")
    return date.fromisoformat(date_str)

def get_everef_file_url(curr_date: date) -> str:
    return f"{BASE_URL}/{curr_date.year}/market-history-{curr_date.isoformat()}.csv.bz2"

def daterange(date_start: date, date_end: date) -> Generator[date]:
    for n in range((date_end - date_start).days + 1):
        yield date_start + timedelta(n)

In [5]:
url = get_everef_file_url(get_date("2026-04-23"))
response = requests.head(url)

response.status_code == codes.OK

True

In [6]:
def validate_content_length_header(item: dict[str, str], response: requests.Response) -> None:
    content_length = response.headers.get("content-length")
    if content_length is None:
        logger.warning("Everef is missing content-length header for %s", item["url"])
        return
    
    if content_length and int(content_length) == 0:
        logger.warning("Everef file has content-length of 0: %s", item["url"])
    else:
        item.update({"content_length": int(content_length)})

def validate_last_modified_header(item: dict[str, str], response: requests.Response) -> None:
    last_modified = response.headers.get("last-modified")
    if last_modified is None:
        logger.warning("Everef is missing last-modified header for %s", item["url"])
        return
    
    try:
        last_modified_dt = parsedate_to_datetime(last_modified)
    except ValueError:
        logger.warning(
            "Everef returned invalid last-modified=%r for %s",
            last_modified,
            item["url"]
        )

    last_modified_iso = last_modified_dt.astimezone(UTC).isoformat()
    item.update({
        "last_modified": last_modified_iso
    })


In [7]:
# MARKET_HISTORY_DTYPES = {
#     "date": "string",
#     "region_id": "int64",
#     "type_id": "int64",
#     "order_count": "int64",
#     "volume": "int64",
#     "average": "float64",
#     "highest": "float64",
#     "lowest": "float64",
# }
MARKET_HISTORY_KEY_COLUMNS = {
    "date": {"nullable": True},
    "region_id": {"nullable": True},
    "type_id": {"nullable": True},
}

In [ ]:
@dlt.resource(name="market_history_urls", selected=False)
def list_file_urls(date_start: date, date_end: date) -> Iterator[dict[str, str]]:
    for curr_date in daterange(date_start, date_end):
        yield {
            "market_date": curr_date.isoformat(), 
            "url": get_everef_file_url(curr_date)
        }

EVEREF_PROBE_CLIENT = requests.Client(raise_for_status=False,
                                      status_codes=(429, 500, 502, 503, 504))

@dlt.transformer(name="market_history_files", selected=False, parallelized=True)
def probe_url_file(item: dict[str, str]) -> Iterator[dict[str, int | str]]:
    try:
        response = EVEREF_PROBE_CLIENT.head(item["url"], allow_redirects=True)
    except requests.RequestException as e:
        logger.warning("Everef probe failed at %s: %s", item["url"], e)
        return
    if response.status_code == 404:
        logger.warning("Everef file missing for %s, %s", item["market_date"], item["url"])
        return
    if response.status_code >= 400:
        logger.warning(
            "Unexpected Everef status HTTP %s for %s",
            response.status_code,
            item["url"]
        )
        return

    validate_content_length_header(item, response)
    validate_last_modified_header(item, response)

    yield item

@dlt.transformer(name="market_history",
                 parallelized=True,
                #  file_format="parquet",
                 write_disposition="merge",
                 primary_key=["date", "region_id", "type_id"],
                 columns=MARKET_HISTORY_KEY_COLUMNS)
def read_market_history_csv(item: dict[str, int | str]) -> Iterator[pd.DataFrame]:
    file_url, market_date = item["url"], item["market_date"]
    ingested_at = datetime.now(UTC).isoformat()

    try:
        chunks = pd.read_csv(
            file_url,
            compression="bz2",
            chunksize=20_000,
            # dtype=MARKET_HISTORY_DTYPES
        )
        key_cols = ["date", "region_id", "type_id"]
        
        for chunk in chunks:
            key_nulls = chunk[key_cols].isna().sum()
            if key_nulls.any():
                logger.warning(
                    "Skipping chunk from %s because primary-key columns contain nulls: %s",
                    file_url,
                    key_nulls.to_dict(),
                )
                continue

            chunk["_ingested_at"] = ingested_at

            yield chunk

    except Exception as e:
        logger.warning("Could not read Everef CSV %s: %s", file_url, e)
        return
    
@dlt.source(name="everef")
def everef_source():
    return list_file_urls(get_date("2025-01-01"), get_date("2025-01-31")) \
            | probe_url_file \
            | read_market_history_csv


In [9]:
pipeline = dlt.pipeline(
    pipeline_name="everef_pipeline_dev",
    # destination="filesystem",
    destination="duckdb",
    dataset_name="everef_history_dev",
    dev_mode=True
)


In [10]:
load_info = pipeline.run(
    everef_source()
)


In [11]:
import duckdb

con = duckdb.connect("everef_pipeline_dev.duckdb", read_only=True)

In [12]:
print(con.sql("SHOW ALL TABLES").df())

              database                                     schema  \
0  everef_pipeline_dev          everef_history_dev_20260508021428   
1  everef_pipeline_dev          everef_history_dev_20260508021428   
2  everef_pipeline_dev          everef_history_dev_20260508021428   
3  everef_pipeline_dev          everef_history_dev_20260508021428   
4  everef_pipeline_dev  everef_history_dev_20260508021428_staging   
5  everef_pipeline_dev  everef_history_dev_20260508021428_staging   

                  name                                       column_names  \
0           _dlt_loads  [load_id, schema_name, status, inserted_at, sc...   
1  _dlt_pipeline_state  [version, engine_version, pipeline_name, state...   
2         _dlt_version  [version, engine_version, inserted_at, schema_...   
3       market_history  [average, date, highest, lowest, order_count, ...   
4         _dlt_version  [version, engine_version, inserted_at, schema_...   
5       market_history  [average, date, highest, lowes

In [ ]:
con.sql("""
        SELECT *
        FROM everef_history_dev_20260508021428.market_history
        WHERE region_id = 10000002
        LIMIT 20
        """).df()

,average,date,highest,lowest,order_count,volume,http_last_modified,region_id,type_id,_source_market_date,_ingested_at
0,13660.00,2025-01-22,13870.00,13600.0,64,426235,2025-05-14T11:02:27Z,10000002,44,2025-01-22,2026-05-08T02:14:34.331611+00:00
1,43.50,2025-01-22,43.95,43.5,125,281070,2025-01-24T11:10:12Z,10000002,184,2025-01-22,2026-05-08T02:14:34.331611+00:00
2,6999000.00,2025-01-22,6999000.00,6999000.0,1,1,2025-05-14T11:02:27Z,10000002,506,2025-01-22,2026-05-08T02:14:34.331611+00:00
3,171961.43,2025-01-22,181000.00,131100.0,45,70,2025-05-14T11:02:27Z,10000002,594,2025-01-22,2026-05-08T02:14:34.331611+00:00
4,160000.00,2025-01-22,160000.00,160000.0,29,29,2025-05-14T11:02:27Z,10000002,812,2025-01-22,2026-05-08T02:14:34.331611+00:00
...,...,...,...,...,...,...,...,...,...,...,...
297624,1504000.00,2025-01-13,1505000.00,1503000.0,2,2,2025-04-07T11:01:10Z,10000002,82464,2025-01-13,2026-05-08T02:14:28.979627+00:00
297625,21624.62,2025-01-13,21630.00,21620.0,3,13,2025-04-07T11:01:10Z,10000002,83251,2025-01-13,2026-05-08T02:14:28.979627+00:00
297626,5389000.00,2025-01-13,9696000.00,5305000.0,122,248,2025-02-12T11:02:03Z,10000002,85258,2025-01-13,2026-05-08T02:14:28.979627+00:00
297627,28000000.00,2025-01-13,28000000.00,28000000.0,1,1,2025-09-15T11:01:39Z,10000002,85519,2025-01-13,2026-05-08T02:14:28.979627+00:00
